In [14]:
import xml.etree.ElementTree as ET
from collections import defaultdict
import pandas as pd
import sqlite3

## **Задания №1 и №2**

In [15]:
tree = ET.parse('a_groups.xml')
root = tree.getroot()

In [16]:
fields = defaultdict(lambda: {
    'types': set(),
    'values': [],
    'lengths': [], 
    'is_null': False,
})

In [17]:
for record in root.findall('another-students-db-a-group'):
    for field in record:
        tag = field.tag
        value = field.text
        is_nil = field.get('nil') == 'true'
        xml_type = field.get('type', 'string')

        fields[tag]['types'].add(xml_type)

        if not is_nil and value is not None:
            fields[tag]['values'].append(value)
            if xml_type == 'string':
                fields[tag]['lengths'].append(len(value))
        else:
            fields[tag]['is_null'] = True

result_df = pd.DataFrame(columns=['Field', 'Type', 'MAX length', 'NOT NULL', 'Unique values'])

for tag, data in fields.items():
    type = ', '.join(data['types'])
    max_length = max(data['lengths']) if data['lengths'] else None
    is_null = False if data.get('is_null', False) else True

    unique_values = set(data['values'])
    if 2 <= len(unique_values) <= 5:
        is_check = f"CHECK IN {set(data['values'])}"
    else:
        is_check = f"CHECK not need, unique values - {len(set(data['values']))}"

    result_df.loc[len(result_df)] =  [tag, type, max_length, is_null, is_check]

result_df

,Field,Type,MAX length,NOT NULL,Unique values
0,id,integer,None,True,"CHECK not need, unique values - 13648"
1,name,string,11,True,"CHECK not need, unique values - 3990"
2,old-name,string,8,False,"CHECK not need, unique values - 400"
3,term-number,integer,None,True,"CHECK not need, unique values - 12"
4,study-year,string,9,True,"CHECK not need, unique values - 15"
5,created-at,dateTime,None,True,"CHECK not need, unique values - 149"
6,updated-at,dateTime,None,True,"CHECK not need, unique values - 149"


## **Задание №3 и №4**

In [18]:
conn = sqlite3.connect('a-groups.db')
cursor = conn.cursor()

In [19]:
cursor.execute('DROP TABLE IF EXISTS university_groups;')

In [20]:
cursor.execute(
'''
CREATE TABLE university_groups (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name VARCHAR(11) NOT NULL,
    old_name VARCHAR(8),
    term_number INTEGER NOT NULL,
    study_year VARCHAR(9) NOT NULL,
    created_at TIMESTAMP NOT NULL,
    updated_at TIMESTAMP NOT NULL
    );
'''
)

In [21]:
pd.read_sql("PRAGMA table_info(university_groups)", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,name,VARCHAR(11),1,None,0
2,2,old_name,VARCHAR(8),0,None,0
3,3,term_number,INTEGER,1,None,0
4,4,study_year,VARCHAR(9),1,None,0
5,5,created_at,TIMESTAMP,1,None,0
6,6,updated_at,TIMESTAMP,1,None,0


In [22]:
insert = '''
INSERT INTO university_groups (name, term_number, study_year, created_at, updated_at)
VALUES (
    'Ф02-05', 
    2,
    '2014/2015', 
    strftime('%Y-%m-%d %H:%M:%S', '2018-05-03T04:18:26+03:00'), 
    strftime('%Y-%m-%d %H:%M:%S', '2018-05-03T04:18:26+03:00')
);
'''

cursor.execute(insert)

In [23]:
pd.read_sql('SELECT * FROM university_groups', conn)

,id,name,old_name,term_number,study_year,created_at,updated_at
0,1,Ф02-05,None,2,2014/2015,2018-05-03 01:18:26,2018-05-03 01:18:26


In [24]:
insert = '''
INSERT INTO university_groups (name, old_name, term_number, study_year, created_at, updated_at)
VALUES (
    'Б14-504',
    'К05-122',
    6,
    '2016/2017',
    strftime('%Y-%m-%d %H:%M:%S', '2018-05-03T04:20:24+03:00'),
    strftime('%Y-%m-%d %H:%M:%S', '2018-05-03T04:20:24+03:00')
);
'''

cursor.execute(insert)

In [26]:
pd.read_sql('SELECT * FROM university_groups', conn)

,id,name,old_name,term_number,study_year,created_at,updated_at
0,1,Ф02-05,None,2,2014/2015,2018-05-03 01:18:26,2018-05-03 01:18:26
1,2,Б14-504,К05-122,6,2016/2017,2018-05-03 01:20:24,2018-05-03 01:20:24


In [27]:
conn.close()